# Deep Agents 1주차 Walkthrough — 한 자리에서 4개 데모 통과시키기

> **부제**: Overview · Quickstart · Customization 을 한 노트북에 합성한다
> **세션**: LangChain AI Agent Master 9주 커리큘럼, **1주차** (발표자 — 송태영)
> **교안**: [`content/01_textbook.md`](../archives/source/01_textbook.md)
> **원본 문서**: [`01-overview_ko.md`](../archives/original_docs/01-overview_ko.md), [`02-quickstart_ko.md`](../archives/original_docs/02-quickstart_ko.md), [`03-customization_ko.md`](../archives/original_docs/03-customization_ko.md)
> **실행 환경**: [`.venv`](../../../.venv) (Python 3.12, deepagents 0.5.6, langchain 1.2.x), [`.env`](../.env)

---

## 이 노트북은 무엇인가

`week1-overview-taeyoung/scripts/` 폴더의 4개 실행 스크립트를 **한 노트북에서 차례로 실행**하면서, 동시에 교안 §0\~§6 의 이론·다이어그램·레퍼런스를 모두 한 자리에 합성한다. 노트북만 봐도 1주차 분량을 이해할 수 있게 한다는 게 목표다.

| 데모 | 스크립트 | 교안 매핑 | 키 의존 |
|---|---|---|---|
| Demo 1 | [`01_quickstart_research_agent.py`](../archives/scripts_py/01_quickstart_research_agent.py) | §3.2~§3.4 | OpenAI(호환) + Tavily |
| Demo 2 | [`02_model_string_swap.py`](../archives/scripts_py/02_model_string_swap.py) | §4.3 길 1 | OpenAI(호환) |
| Demo 3 | [`03_model_object_ollama.py`](../archives/scripts_py/03_model_object_ollama.py) | §4.3 길 2 | 로컬 Ollama (tools 지원 모델) |
| Demo 4 | [`04_custom_system_prompt.py`](../archives/scripts_py/04_custom_system_prompt.py) | §4.4 | OpenAI(호환) |

---

## 학습 목표 (이 노트북을 다 돌리고 나면)

| # | 목표 |
|---|---|
| L1 | Deep Agent 가 무엇인지 한 문장으로 정의하고, 4대 능력을 도구 이름과 매칭시킬 수 있다 |
| L2 | `create_deep_agent()` 한 호출로 첫 에이전트를 5줄 이내로 짤 수 있다 |
| L3 | `agent.invoke()` 가 "백그라운드에서 일하는 동안" 일어나는 5단계를 설명할 수 있다 |
| L4 | Model · System Prompt · Tools 세 다이얼을 자기 도메인에 맞춰 돌릴 수 있다 |
| L5 | `create_agent` / 직접 짠 LangGraph / `create_deep_agent` 결정 표를 자기 문제에 적용할 수 있다 |

---

## 노트북 사용법

1. **커널 선택**: `.venv` 의 Python 3.12 (방금 설치한 ipykernel)
2. **`.env` 위치 확인**: `week1-overview-taeyoung/.env` (이 노트북의 부모 디렉토리). `find_dotenv()` 가 자동으로 찾는다
3. **셀 단위 실행** (`Shift+Enter`): 위에서부터 차례대로. 각 데모 셀은 독립이지만, §1 환경 점검은 먼저 한 번 통과해야 한다
4. **예상 소요 시간**: 30\~45분 (Demo 1 의 Tavily 검색이 가장 길다 — 30~90초)
5. **트러블슈팅**: §10 부록의 표 참조

> **주의 — 타임아웃**: clipproxyapi(`gpt-5.x`) 또는 OpenAI 직접 호출 시 도구 호출이 많이 발생하면 한 번의 `invoke` 가 1\~3분 걸릴 수 있다. 중간에 멈추면 셀을 다시 실행한다.


## 섹션 지도

| 노트북 § | 다루는 내용 | 교안 매핑 |
|---|---|---|
| §1 | 환경 점검 — venv, `.env`, imports | (셋업) |
| §2 | 왜 Deep Agent 인가 — Vanilla 한계 + 3사례 + 라이브러리화 | 교안 §1 |
| §3 | 4대 내장 능력 — Planning · Filesystem · Subagent · Memory · execute | 교안 §2 |
| §4 | **Demo 1** Quickstart 리서치 에이전트 (§3.2~§3.4) | 교안 §3 + 스크립트 01 |
| §5 | 청사진 (17 파라미터 → 3 다이얼) + **Demo 2/3/4** | 교안 §4 + 스크립트 02·03·04 |
| §6 | 언제 쓰나 — 의사결정 표 + 플로우차트 | 교안 §5 |
| §7 | 교안 너머 — 미들웨어 체인, LangSmith 트레이싱, 통합 모델 표 | 교안 §6 + 보강 |
| §8 | 용어집 (15 용어) | 교안 부록 A |
| §9 | 레퍼런스 & 추가 자료 (URL 25+) | 외부 |
| §10 | 부록 — `.env` 키 의미, 트러블슈팅 표 | scripts/README.md |


## §1. 환경 점검 — Setup & Imports

이 노트북이 가정하는 환경은 직전 세션에서 검증돼 있다.

- **venv**: `/home/restful3/workspace/langchain-docs/deep-agents/.venv` (Python 3.12)
- **`.env` 위치**: `week1-overview-taeyoung/.env` — 4개 스크립트가 모두 `find_dotenv()` 로 이 파일을 자동으로 찾는다
- **모델 라우팅**:
  - OpenAI 호환 → clipproxyapi 로컬 프록시 (`http://localhost:8317/v1`, `gpt-5.4-mini`) — Demo 1 / 2 / 4
  - 로컬 Ollama → tools 지원 모델 (`PetrosStav/gemma3-tools:12b`) — Demo 3
- **검색 키**: Tavily (Demo 1)

다음 셀들로 환경이 정상인지 한 번 점검한다.


In [ ]:
# 1) 핵심 라이브러리 버전을 한 번 확인한다.
#    deepagents / langchain / langgraph / tavily / dotenv / ollama 모두 설치돼 있어야 한다.
#    (직전 세션에서 .venv/bin/pip install -r requirements.txt 로 설치 완료)

import importlib.metadata as meta

for pkg in [
    "deepagents",
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-openai",
    "langchain-ollama",
    "tavily-python",
    "python-dotenv",
]:
    try:
        version = meta.version(pkg)
    except meta.PackageNotFoundError:
        version = "MISSING"
    print(f"{pkg:24s}  {version}")


In [ ]:
# 2) .env 파일 위치를 확인하고 환경변수를 메모리에 적재한다.
#    find_dotenv() 가 현재 작업 디렉토리에서 위로 올라가며 .env 를 찾는다.
#    노트북이 scripts/ 에 있으니 한 단계 위(week1-overview-taeyoung/)의 .env 가 잡힌다.

import os
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv()
print(f".env 위치: {dotenv_path}")

loaded = load_dotenv(dotenv_path)
print(f"load_dotenv 결과: {loaded}\n")

# 어떤 키들이 잡혔는지 (값은 마스킹) 확인.
for key in [
    "OPENAI_API_KEY",
    "OPENAI_BASE_URL",
    "DEEPAGENT_MODEL",
    "TAVILY_API_KEY",
    "OLLAMA_MODEL",
]:
    val = os.environ.get(key, "")
    if not val:
        masked = "(empty)"
    elif "URL" in key or "MODEL" in key:
        masked = val  # URL/모델명은 그대로 보여도 안전
    else:
        masked = val[:6] + "..." + val[-3:] if len(val) > 12 else "***"
    print(f"  {key:18s} = {masked}")


In [ ]:
# 3) 핵심 import.
#    이 4개가 깨끗이 import 되면 본 노트북의 모든 데모가 실행 가능한 상태다.

from langchain.chat_models import init_chat_model  # 모델 인스턴스화 진입점
from deepagents import create_deep_agent           # 라이브러리 메인 팩토리
from tavily import TavilyClient                    # Demo 1 의 검색 클라이언트

print("imports OK")
print("create_deep_agent =", create_deep_agent)


## §2. 왜 Deep Agent 인가 — 교안 §1 압축

> **한 줄**: 「LLM 한 번 + 도구 몇 개 + while 루프」 패턴은 짧은 작업에는 잘 동작하지만, 단계가 많아지면 무너진다.

LangChain `create_agent` 로 만들어 본 에이전트를 떠올려보자. 도구 호출이 가능한 모델, 도구 리스트, 시스템 프롬프트 한 장 — 이 셋만 묶으면 동작한다. 짧은 질의응답이나 함수 호출 한두 번이 끝인 작업에는 충분히 강력하다.

문제는 **작업이 길어질 때** 시작된다. Anthropic 의 엔지니어링 글은 에이전트가 "사람처럼" 일하게 하려면 다음 네 단계 루프가 반복돼야 한다고 정리한다 — **gather context → take action → verify work → repeat**. 이 루프를 단순 ReAct 한 줄로 돌리면 세 가지 지점에서 깨진다.

### 표 1: Vanilla 에이전트가 깨지는 세 지점

| 깨지는 지점 | 무엇이 일어나는가 |
|---|---|
| 컨텍스트 오버플로 | 검색 결과·문서·대화 이력이 누적돼 모델 컨텍스트 윈도우를 잡아먹는다. 30회 검색 이후엔 시스템 프롬프트마저 잘려 나간다. |
| 계획의 부재 | 모델이 "다음 단계" 를 매 턴 즉흥으로 결정. 중간에 길을 잃거나 같은 검색을 반복한다. |
| 위임 불가 | 한 모델 인스턴스가 모든 컨텍스트를 들고 있으니, 무거운 하위 작업이 메인 흐름을 오염시킨다. |


### 그림 1: Vanilla vs Deep Agent

<img src="../content/figs/fig01_vanilla_vs_deep_agent.svg" alt="Vanilla vs Deep Agent" width="900"/>

> 좌측은 LangChain `create_agent` 로 만든 단순 에이전트의 흐름이다. User → LLM ↔ Tool 의 한 루프가 응답을 만든다. 단계가 다섯 개 정도까지는 잘 동작하지만, 그 이상 누적되면 표 1 의 세 지점이 한꺼번에 무너진다. 우측은 `create_deep_agent` 의 흐름이다. LLM 호출 사이에 **미들웨어 체인** 한 켜가 들어가서, 매 turn 마다 (1) 계획을 갱신하고, (2) 큰 결과를 가상 파일시스템에 오프로드하고, (3) 무거운 하위 작업은 서브에이전트로 위임한다. 모델이 사용하는 도구 풀도 **빌트인 9종 + 사용자 도구** 로 두꺼워진다.

Vanilla 패턴은 **컨텍스트를 외부화** 할 곳도, **계획을 명시화** 할 자리도, **컨텍스트를 격리** 할 경계도 없다. 단계 5개까지는 견디지만 50개에서는 무너진다.


### §2.2 문제를 해결한 3가지 사례 — 같은 패턴으로 수렴

| 시스템 | 출시 주체 | 노출 방식 | 핵심 도구 카테고리 |
|---|---|---|---|
| **Claude Code** | Anthropic | 코드 어시스턴트 | 파일시스템, agentic search, **서브에이전트**, **컴팩션**, bash, code generation, MCP |
| **OpenAI Deep Research** | OpenAI | 자율 리서치 에이전트 | Clarification → Decomposition → Iterative Search → Multi-format Reading → Synthesis (5단계) |
| **Manus** | Manus 팀 | 자율 에이전트 | CodeAct (Python 스크립트 생성·샌드박스 실행), `todo.md` 진행 추적, 격리된 서브에이전트 |

#### Claude Code (Anthropic)

> *"give your agents a computer, allowing them to work like humans do."* — Anthropic Engineering Blog

사람이 컴퓨터에서 일할 때 쓰는 도구 카테고리를 그대로 모델에 준다는 발상이다. 파일시스템·서브에이전트·컴팩션은 deepagents 의 4대 능력 중 셋과 그대로 겹친다.

#### OpenAI Deep Research

핵심은 두 번째와 세 번째 단계다 — 큰 질문을 작은 작업으로 **분해(decomposition)** 하고, 각 작업을 ReAct 루프(Plan → Act → Observe) 로 돌린다. 한 작업당 평균 30~60회 검색, 120~150 페이지 읽기, 추론 150~200 iteration. 이 규모에서는 컨텍스트 외부화와 계획 명시화 없이 동작 자체가 불가능하다.

#### Manus

Claude 3.5/3.7 + 파인튜닝된 Qwen 위에 만들어진 자율 에이전트로 **CodeAct** 패러다임을 쓴다 — 도구 호출 대신 모델이 짧은 Python 스크립트를 생성하면 샌드박스가 실행한다. 진행 상황은 가상 파일시스템 안의 `todo.md` 에 기록된다.


### 그림 2: 세 시스템이 같은 디자인 축으로 수렴

<img src="../content/figs/fig02_three_systems_common_pattern.svg" alt="Three Systems Common Pattern" width="900"/>

> **세 시스템의 공통 분모** (Harrison Chase 의 정리, [Deeper LangChain Agents 출시 발표](https://blog.langchain.com/deep-agents/)):
>
> *"Deep agents are agents that perform **planning**, use **sub agents**, have access to a **file system**, and have a **detailed prompt**."*

세 시스템은 출시 주체도 다르고(Anthropic / OpenAI / Manus 팀), 노출 방식도 다르고(코드 어시스턴트 / 자율 리서치 / 자율 에이전트), 구현 디테일도 다르다. 그러나 **무엇이 들어 있는가** 를 한 발짝 떨어져서 보면 같은 네 가지로 수렴한다 — 계획을 명시 기록하는 자리, 큰 컨텍스트를 빼두는 자리, 무거운 하위 작업을 격리해 위임하는 자리, 그리고 이 모든 것을 모델이 "사용하게" 만드는 한 장의 잘 짜인 시스템 프롬프트.


### §2.3 라이브러리화 — 새 프레임워크가 아니라 한 켜의 미들웨어

핵심 결정은 두 가지다.

1. **새 프레임워크가 아니라 라이브러리**로. LangGraph 를 그대로 깔고 그 위에 미들웨어 한 켜를 올린다 — `create_deep_agent()` 의 반환 타입은 `CompiledStateGraph` 다. 즉 지금 쓰는 LangGraph 도구·관측성·배포 인프라가 그대로 쓰인다.
2. **선택을 강요하지 않는다**. 4대 능력은 모두 미들웨어 형태로 들어 있어 켜고 끌 수 있다 — TodoListMiddleware, FilesystemMiddleware, SubAgentMiddleware, SummarizationMiddleware 등이 LangChain core 의 prebuilt middleware 카탈로그(16종) 안에 함께 정리돼 있다.


### 그림 3: deepagents 의 스택 위치

<img src="../content/figs/fig03_stack_position.svg" alt="Stack Position" width="900"/>

> 가장 아래에는 LangGraph 가 있다 — `StateGraph`, 체크포인터(thread 상태 영속), Store(thread 횡단 영속) 같은 그래프 실행·상태 관리 인프라를 제공한다. 그 위에 LangChain 이 올라가 ChatModel · Runnable · 도구 호출 표준을 통일한다. **deepagents 는 그 위 한 켜** 다 — 4대 능력을 미들웨어 형태로 박아 넣고 BASE_AGENT_PROMPT 한 장을 합성한다. `create_deep_agent()` 가 반환하는 것이 그냥 LangGraph 의 `CompiledStateGraph` 라는 점이 결정적이다 — 지금 LangGraph 로 이미 돌리고 있는 관측(LangSmith)·배포(LangGraph Platform)·체크포인터·Store 가 그대로 쓰인다.


## §3. 4가지 내장 능력 — 교안 §2 정밀

> **한 줄**: 비서에게 「할 일 목록 · 노트 · 인턴 · 일기장」 을 쥐여 준다 — Planning · Filesystem · Subagent · Long-term Memory.


### 그림 4: 4대 능력의 비유 — Deep Agent 는 잘 갖춰진 비서

<img src="../content/figs/fig04_four_capabilities_metaphor.svg" alt="Four Capabilities Metaphor" width="900"/>

| 능력 | 비유 | 도구 이름 | 미들웨어 |
|---|---|---|---|
| **Planning** | 할 일 목록 | `write_todos` | TodoListMiddleware |
| **Filesystem** | 노트 (책상 위 문서) | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep` | FilesystemMiddleware |
| **Subagents** | 인턴 (위임할 손) | `task` | SubAgentMiddleware |
| **Long-term Memory** | 일기장 (해를 넘기는 기록) | `write_file` / `read_file` (with `/memories/` prefix) | CompositeBackend (FilesystemMiddleware + LangGraph Store) |

이 비유에 5번째 도구 `execute` 를 더하면 도구 풀이 9종으로 완성된다 — `write_todos`, `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, `execute`, `task`. 사용자 도구는 이 위에 추가된다.


### §3.1 Planning — `write_todos`

> **한 줄**: 모델이 "다음 단계" 를 즉흥으로 정하는 대신, 자기 계획을 파일로 적어놓고 그 파일을 보며 일한다.

내장 `write_todos` 도구는 **TodoListMiddleware** 가 제공한다. 모델이 작업 시작 시 할 일을 4~7개 항목으로 쪼개어 todo 리스트로 기록하고, 한 항목을 끝낼 때마다 `[x]` 표시를 붙이며 진행 상황을 추적한다.

`write_todos` 가 만들어내는 출력은 보통 다음 모양이다:

```text
- [x] 1. langgraph 의 핵심 추상 4가지 정의
- [x] 2. 각 추상이 LangChain 과 어떻게 다른지 정리
- [ ] 3. 코드 예제로 StateGraph + Send 패턴 시연
- [ ] 4. 보고서 초안 작성
```

#### 흥미로운 사실 — `write_todos` 는 사실 no-op 이다

> *"Claude Code uses a Todo list tool. Funnily enough — this doesn't do anything! It's basically a no-op."* — Harrison Chase, [Deep Agents 출시 발표](https://blog.langchain.com/deep-agents/)

진짜로 어딘가에 todo list 가 저장되고 추적되는 게 아니라, 모델이 "할 일을 한 번 적어두는 행위" 자체가 컨텍스트에 계획을 남겨 다음 turn 의 모델이 자기 계획을 잊지 않게 만드는 **컨텍스트 엔지니어링 트릭**이다. 도구가 하는 일은 모델이 적은 todo 텍스트를 그대로 state 의 `todos` 슬롯에 박아두는 것뿐 — 그래도 효과가 큰 이유는 다음 turn 의 시스템 프롬프트에 그 todo 가 다시 들어가 모델이 자기 계획을 매 turn 마주하기 때문이다.

발상의 뿌리: Claude Code 의 BASE_AGENT_PROMPT 에 명시된 **Understand → Act → Verify** 3단 워크플로 + Manus 의 `todo.md` 패턴.


### §3.2 Filesystem — 가상 파일시스템 4 도구

> **한 줄**: 큰 검색 결과·중간 산출물을 모델 컨텍스트에서 빼서 가상 파일시스템에 적어두고, 필요할 때만 읽는다.

#### 표 2: Filesystem 미들웨어가 추가하는 4개 도구

| 도구 | 시그니처 | 역할 |
|---|---|---|
| `ls()` | — | 가상 파일시스템의 파일 일람 |
| `read_file(path, offset, limit)` | path: str, offset: int, limit: int | 큰 파일을 페이지 단위로 읽기 |
| `write_file(path, content)` | path: str, content: str | 새 파일 작성 |
| `edit_file(path, old_str, new_str)` | path: str, old_str: str, new_str: str | 부분 치환 (줄 단위 정확 매칭) |

여기에 `glob`, `grep` 같은 검색 도구도 함께 노출된다.

#### Backend 추상 — 같은 도구, 다른 저장소

이 가상 파일시스템은 **Backend** 라는 추상으로 한 단 더 깊어진다.

| Backend | 수명 | 사용 시점 |
|---|---|---|
| `StateBackend` (기본값) | 한 thread (한 invoke 사이클) | 임시 결과물, 대화 한 번 안에서만 필요한 노트 |
| `StoreBackend` | 모든 thread (LangGraph Store) | 사용자 프로필, 대화 간 공유될 영구 메모 |
| `CompositeBackend` | 경로 prefix 로 라우팅 | `/memories/` → Store, 그 외 → State (혼합 시나리오) |


### 그림 5: Filesystem Backend 라우팅 — 같은 도구, 다른 저장소

<img src="../content/figs/fig05_filesystem_backend_routing.svg" alt="Filesystem Backend Routing" width="900"/>

> 모델은 `write_file` 한 도구만 알지만, 그 호출이 실제 어디에 저장되는지는 **경로(prefix)** 가 결정한다. `CompositeBackend` 가 `/memories/` 로 시작하는 경로는 `StoreBackend` (LangGraph Store, thread 횡단 영구) 로, 그 외 경로는 `StateBackend` (LangGraph state, 한 thread 안에서만) 로 라우팅한다.

**왜 이게 컨텍스트 관리인가** — 검색 결과를 그대로 모델에 다 박으면 한 턴에 5\~10K 토큰이 먹힌다. 검색 결과를 `write_file("results/search_01.md", ...)` 로 빼두고, 본문에는 "결과는 search_01.md 에 저장됨" 한 줄만 남기면 모델 컨텍스트는 간결하게 유지되고, 다음 턴에 필요한 부분만 `read_file` 로 끌어올 수 있다. 이게 Anthropic 이 말한 **compaction** 의 LangGraph 버전이다.


### §3.3 Subagents — `task`

> **한 줄**: 무거운 하위 작업은 격리된 서브에이전트에 위임 — 메인 컨텍스트가 깨끗해진다.

`SubagentMiddleware` 가 `task` 도구 하나를 더한다. 메인 에이전트는 `task` 를 호출해 하위 작업을 띄울 수 있고, 디폴트로 **`general-purpose`** 라는 범용 서브에이전트가 항상 존재한다. 이 디폴트 서브에이전트는 메인 에이전트와 동일한 도구 풀을 가진 단순 복제다 — 실질적으로 "메인의 똑같은 카피본인데 컨텍스트만 격리된 또 한 명" 인 셈이다.

#### 격리되는 두 차원

| 격리 차원 | 효과 |
|---|---|
| **컨텍스트** | 서브에이전트는 자기만의 메시지 히스토리·도구 로그를 가지고 일한다. 메인 에이전트에게는 최종 결과(짧은 요약 또는 문서 한 편)만 돌아온다 |
| **권한** | 서브에이전트마다 도구 풀을 다르게 줄 수 있다. 예: 코드 실행 권한은 코드 서브에이전트에만, 검색은 리서치 서브에이전트에만 |

#### `task` 의 3가지 형태

API 레퍼런스는 서브에이전트를 세 형식으로 받는다고 명시한다:

- **`SubAgent`** — 선언형 동기 (가장 일반적). dict 또는 Pydantic 모델로 정의
- **`CompiledSubAgent`** — 이미 컴파일된 LangGraph runnable 을 그대로 등록 (서브에이전트를 별도로 LangGraph 로 짠 경우)
- **`AsyncSubAgent`** — 원격/백그라운드 실행 (예: 다른 LangGraph 서버에 RPC 호출). `AsyncSubAgentMiddleware` 가 처리

이 글에서는 가장 일반적인 `SubAgent` 까지만 짚는다 — async/compiled 디테일은 별도 글에서 다뤄진다.


### 그림 6: 서브에이전트의 컨텍스트 격리

<img src="../content/figs/fig06_subagent_isolation.svg" alt="Subagent Isolation" width="900"/>

> 좌측 메인 에이전트는 사용자와 대화하며 `task("스펙 분석", ...)` 한 줄로 하위 작업을 위임한다. 그 한 줄 뒤에서 우측 서브에이전트가 자체 컨텍스트(자체 메시지 히스토리·자체 도구 풀·자체 권한)로 검색 30회를 돌리고 한 단락 짜리 결과만 메인에 회신한다. 메인 컨텍스트에는 `task` 호출 한 줄과 그 결과 한 단락만 남는다 — 30회 검색의 모든 turn 기록은 보이지 않는다.


### §3.4 Long-term Memory — LangGraph Store

> **한 줄**: 대화/스레드를 넘어 살아남는 기억은 LangGraph Store 가 맡는다 — Filesystem 의 `/memories/` prefix 가 그곳으로 라우팅된다.

LangGraph 에는 두 종류의 영속 계층이 있다.

#### 표 3: LangGraph 의 영속 계층 두 종류

| 계층 | 수명 | deepagents 매핑 |
|---|---|---|
| Checkpointer | 한 thread (한 대화) | 단기 메모리, `agent.invoke` 의 중간 상태 영속 (재호출 시 이어가기) |
| Store | 모든 thread 가 공유 | **장기 메모리** — 다른 대화에서도 읽힘 |

`CompositeBackend(state_backend=..., store_backend=..., prefix="/memories/")` 한 줄로 가상 파일시스템의 `/memories/` 아래 경로는 Store 에, 나머지는 State 에 저장된다. 결과적으로 모델은 **같은 도구 (`write_file`, `read_file`)** 로 단기·장기 메모리를 다룬다 — 차이는 경로뿐이다.

> 이 부분의 깊이 (CompositeBackend 구성, Store API, namespacing) 는 별도 글로 양보한다 — 다음 발표자(정훈) 의 「컨텍스트·메모리·스킬」 에서 다뤄진다.


### §3.5 그 외 — `execute` (5번째 빌트인 도구)

> **한 줄**: 셸 명령을 실행하는 빌트인 도구. 4대 능력 어디에도 직접 매핑되지 않으며, 적절한 sandbox 백엔드가 있을 때만 동작한다.

빌트인 도구 9종 중 5번째 카테고리에 해당하는 `execute` 는 §3.1\~§3.4 의 4대 능력 어디에도 들어가지 않는다. API 레퍼런스가 명시한다:

> *"The `execute` tool allows running shell commands if the backend implements `SandboxBackendProtocol`. For non-sandbox backends, the `execute` tool will return an error message."*

기본 `StateBackend` 에서 `execute` 를 호출하면 모델이 받는 건 에러 메시지뿐이다. 셸을 실제로 돌리려면 sandbox 능력을 가진 백엔드(예: 컨테이너 격리)를 끼워야 한다.

> 백엔드·샌드박스의 실제 구성은 별도 글로 양보한다 — 종훈L 발표의 「백엔드·샌드박스·권한」 주제.


## §4. Demo 1: Quickstart 리서치 에이전트 — 교안 §3 + 스크립트 01

> **한 줄**: 검색 도구 한 개 + 시스템 프롬프트 한 장 + `create_deep_agent` 한 줄 — Quickstart 의 코어는 정말로 다섯 줄이다.

원본 [`02-quickstart_ko.md`](../archives/original_docs/02-quickstart_ko.md) 의 1\~5단계와 [`scripts/01_quickstart_research_agent.py`](../archives/scripts_py/01_quickstart_research_agent.py) 를 한 자리에서 풀어보고, 실제로 invoke 해서 응답까지 받는다.

### `agent.invoke()` 백그라운드 5단계


<img src="../content/figs/fig07_invoke_five_phases.svg" alt="Invoke Five Phases" width="900"/>

> 사용자 질문이 들어가면 deep agent 는 다섯 단계를 거쳐 응답을 만든다.
> ① 먼저 `write_todos` 로 작업을 4\~7개 항목으로 분해한다.
> ② 그 항목 하나씩을 `internet_search` 같은 도구로 처리한다.
> ③ 검색 결과가 컨텍스트를 잡아먹기 시작하면 `write_file`/`read_file` 로 오프로드한다.
> ④ 한 항목이 너무 무거우면 `task` 로 서브에이전트에 위임한다.
> ⑤ todo 의 모든 항목이 `[x]` 가 되면 결과를 종합한다.
>
> 다섯 단계가 한 turn 안에서 모두 일어나는 게 아니라 **여러 turn 에 걸쳐** 미들웨어 체인이 매번 같은 순서로 돌아간다 — 모델은 그저 도구가 거기 있다고 인식할 뿐 단계 자체는 보지 못한다.

이제 실제로 만들어 본다.


In [ ]:
# §4.1 — 모델 인스턴스화.
#
# init_chat_model 은 LangChain 의 모델 진입점이다. 형식은 두 가지:
#   1) 문자열  : init_chat_model("openai:gpt-5.4-mini")
#   2) 객체    : init_chat_model(model="...", model_provider="ollama", temperature=0)
#
# 우리 .env 는 OPENAI_BASE_URL 을 clipproxyapi 로컬 프록시로 두고 있어,
# init_chat_model 의 base_url kwarg 로 그 값을 그대로 흘려준다.
# (인자가 ChatOpenAI 의 base_url 로 forward 된다)

import os
from langchain.chat_models import init_chat_model

MODEL_NAME = os.environ.get("DEEPAGENT_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")  # None 이면 OpenAI 직접

# OPENAI_BASE_URL 이 비어 있으면 OpenAI 공식 엔드포인트, 채워져 있으면 호환 프록시.
extra = {"base_url": OPENAI_BASE_URL} if OPENAI_BASE_URL else {}
model = init_chat_model(f"openai:{MODEL_NAME}", **extra)

print(f"모델: openai:{MODEL_NAME}")
print(f"base_url: {OPENAI_BASE_URL or '(OpenAI 공식)'}")
print(f"인스턴스: {type(model).__name__}")


In [ ]:
# §4.2 — Tavily 검색 도구 정의.
#
# Tavily 클라이언트 한 개를 만들고 함수 하나로 감싸면 끝이다.
# 함수 시그니처(타입 힌트 포함) + docstring 이 자동으로 LangChain 도구 스키마가 된다 —
# 이것이 deepagents 가 "사용자 도구를 그냥 함수로 받는" 이유.

from typing import Literal
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""  # ← 이 docstring 이 곧 도구의 description.
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

# 함수 자체가 그대로 도구 — 별도 등록 없이 create_deep_agent 의 tools= 에 넘기면 된다.
print("도구 정의 OK:", internet_search.__name__, "—", internet_search.__doc__)


In [ ]:
# §4.3 — research_instructions 시스템 프롬프트 + create_deep_agent 호출.
#
# system_prompt= 은 라이브러리의 BASE_AGENT_PROMPT 를 "교체" 하는 것이 아니라
# 그 앞(USER 자리)에 prepend 된다. (§5.3 그림 9 참조)
# 그래서 이 짧은 도메인 프롬프트가 BASE 의 Understand→Act→Verify 워크플로 위에 얹힌다.

from deepagents import create_deep_agent

research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.
You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    model=model,                            # §4.1 에서 만든 모델 인스턴스
    tools=[internet_search],                # 사용자 도구 1개. 빌트인 9종은 자동
    system_prompt=research_instructions,    # USER 슬롯에 들어가서 BASE 위에 합성됨
)

# 반환 타입은 LangGraph 의 CompiledStateGraph — .invoke / .stream / .ainvoke 모두 사용 가능.
print(f"agent 타입: {type(agent).__name__}")
print(f"노드 수: {len(agent.nodes)}")


In [ ]:
# §4.4 — agent.invoke() 호출. 4대 능력이 백그라운드에서 작동하며 응답을 만든다.
#
# 입력 형식: {"messages": [{"role": "user", "content": "..."}]}  ← LangGraph AgentState
# (같은 dict 에 files=, todos= 도 넣을 수 있고, 응답에서도 같은 키로 받는다.)
#
# clipproxyapi 의 gpt-5.4-mini 가 LangGraph 위에서 도구 호출을 도는 데
# 보통 5~30초 정도 걸린다. 검색 도구가 발동되면 그만큼 더 걸린다.

result = agent.invoke({
    "messages": [{"role": "user", "content": "What is langgraph?"}]
})

print("=" * 70)
print("최종 응답:")
print("=" * 70)
print(result["messages"][-1].content)


In [ ]:
# §4.5 — 결과 인스펙션 (교안에 없는 디버깅 시점).
#
# result 안에는 최종 메시지 외에도 4대 능력의 흔적이 함께 들어 있다:
#   - messages : 모든 turn 의 메시지 (User / AI / ToolCall / ToolMessage)
#   - files    : write_file 로 만든 가상 파일들 (FilesystemMiddleware)
#   - todos    : write_todos 가 박은 계획 트레이스 (TodoListMiddleware)

print("=" * 70)
print(f"메시지 수: {len(result['messages'])}")
print("=" * 70)
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    # 도구 호출이 있는 AIMessage 는 짧게 요약, 그 외는 처음 100자만.
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        names = [tc["name"] for tc in msg.tool_calls]
        preview = f"<tool_calls: {names}>"
    else:
        content = getattr(msg, "content", "")
        if isinstance(content, list):
            content = str(content)
        preview = (content[:100] + "...") if len(content) > 100 else content
    print(f"  [{i:2d}] {msg_type:20s} {preview}")

print()
print("=" * 70)
print("가상 파일시스템:")
print("=" * 70)
files = result.get("files", {})
if files:
    for path, content in files.items():
        size = len(content) if isinstance(content, str) else "?"
        print(f"  {path}  ({size}자)")
else:
    print("  (비어 있음 — 단순 질의는 오프로드를 발동시키지 않을 수 있음)")

print()
print("=" * 70)
print("Todo 리스트:")
print("=" * 70)
todos = result.get("todos", [])
if todos:
    for t in todos:
        print(f"  {t}")
else:
    print("  (비어 있음 — 짧은 질의는 write_todos 가 발동되지 않을 수 있음)")


### 미들웨어 체인의 실제 순서 (교안 §3.4 보강)

API 레퍼런스가 명시한 base stack 의 실제 미들웨어 호출 순서는 다음과 같다.

| # | 미들웨어 | 역할 | 활성 조건 |
|---|---|---|---|
| 1 | `TodoListMiddleware` | 매 turn 시작 시 `todos` 슬롯을 시스템 프롬프트에 주입 | 항상 |
| 2 | `SkillsMiddleware` | `skills=` 인자가 있으면 스킬을 활성화 | `skills=` 제공 시 |
| 3 | `FilesystemMiddleware` | `ls`/`read_file`/`write_file`/`edit_file`/`glob`/`grep` 도구 부착 + state 의 `files` 슬롯 갱신 | 항상 |
| 4 | `SubAgentMiddleware` | `task` 도구 부착 + 등록된 서브에이전트 라우팅 | 항상 (디폴트로 `general-purpose` 가 있음) |
| 5 | `SummarizationMiddleware` | 메시지가 너무 길어지면 요약본으로 압축 | 컨텍스트 임계 초과 시 |
| 6 | `PatchToolCallsMiddleware` | 모델별 도구 호출 형식 차이 보정 | 항상 |
| 7 | `AsyncSubAgentMiddleware` | 비동기 서브에이전트 라우팅 | `AsyncSubAgent` 등록 시 |
| 8 | (Anthropic 모델 한정) `prompt_caching` | 시스템 프롬프트 캐싱 hint | Anthropic 모델 |

체인 안에서 모든 turn 은 위에서 아래로 흐른 뒤 모델 호출로 들어가고, 도구 실행이 끝나면 같은 순서로 다시 위로 돈다. 위 그림 7 의 5단계는 **개념적 흐름** 이며, 실제 미들웨어 chain 순서와 정확히 1:1 대응하는 것은 아니다.


### 그림 (추가): 미들웨어 체인 — 모델 호출을 둘러싼 케이크 층

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 460" width="900" style="background:#fff;font-family:'Inter',sans-serif">
  <defs>
    <marker id="arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="#444"/>
    </marker>
  </defs>
  <!-- input -->
  <rect x="20" y="200" width="120" height="60" rx="10" fill="#FFF7ED" stroke="#9A3412" stroke-width="2"/>
  <text x="80" y="225" text-anchor="middle" font-size="14" font-weight="700" fill="#7C2D12">User invoke</text>
  <text x="80" y="245" text-anchor="middle" font-size="11" fill="#7C2D12">{"messages": [...]}</text>
  <!-- arrow -->
  <line x1="140" y1="230" x2="170" y2="230" stroke="#444" stroke-width="2" marker-end="url(#arr)"/>
  <!-- middleware stack box -->
  <rect x="180" y="40" width="380" height="380" rx="14" fill="#F0F9FF" stroke="#0369A1" stroke-width="2"/>
  <text x="370" y="65" text-anchor="middle" font-size="14" font-weight="700" fill="#0369A1">Middleware Chain</text>
  <!-- stacked middlewares -->
  <g font-size="12" fill="#0C4A6E">
    <rect x="200" y="80" width="340" height="34" rx="6" fill="#E0F2FE" stroke="#0369A1"/>
    <text x="370" y="102" text-anchor="middle" font-weight="600">1. TodoListMiddleware</text>
    <rect x="200" y="120" width="340" height="34" rx="6" fill="#E0F2FE" stroke="#0369A1"/>
    <text x="370" y="142" text-anchor="middle" font-weight="600">2. SkillsMiddleware (선택)</text>
    <rect x="200" y="160" width="340" height="34" rx="6" fill="#E0F2FE" stroke="#0369A1"/>
    <text x="370" y="182" text-anchor="middle" font-weight="600">3. FilesystemMiddleware</text>
    <rect x="200" y="200" width="340" height="34" rx="6" fill="#E0F2FE" stroke="#0369A1"/>
    <text x="370" y="222" text-anchor="middle" font-weight="600">4. SubAgentMiddleware</text>
    <rect x="200" y="240" width="340" height="34" rx="6" fill="#E0F2FE" stroke="#0369A1"/>
    <text x="370" y="262" text-anchor="middle" font-weight="600">5. SummarizationMiddleware</text>
    <rect x="200" y="280" width="340" height="34" rx="6" fill="#E0F2FE" stroke="#0369A1"/>
    <text x="370" y="302" text-anchor="middle" font-weight="600">6. PatchToolCallsMiddleware</text>
    <rect x="200" y="320" width="340" height="34" rx="6" fill="#E0F2FE" stroke="#0369A1"/>
    <text x="370" y="342" text-anchor="middle" font-weight="600">7. AsyncSubAgentMiddleware (선택)</text>
    <rect x="200" y="360" width="340" height="34" rx="6" fill="#FEF3C7" stroke="#92400E"/>
    <text x="370" y="382" text-anchor="middle" font-weight="700" fill="#92400E">→ Model.invoke(messages)</text>
  </g>
  <!-- arrow out -->
  <line x1="560" y1="230" x2="600" y2="230" stroke="#444" stroke-width="2" marker-end="url(#arr)"/>
  <!-- output -->
  <rect x="610" y="160" width="260" height="140" rx="10" fill="#F0FDF4" stroke="#166534" stroke-width="2"/>
  <text x="740" y="185" text-anchor="middle" font-size="14" font-weight="700" fill="#14532D">AgentState</text>
  <g font-size="11" fill="#14532D">
    <text x="625" y="210">• messages: [...]  (대화 누적)</text>
    <text x="625" y="230">• files: {...}    (가상 FS)</text>
    <text x="625" y="250">• todos: [...]    (계획 트레이스)</text>
    <text x="625" y="280" font-style="italic">↻ 다음 turn 으로 같은 chain</text>
  </g>
</svg>

> 한 invoke 당 위 체인이 **여러 turn 반복**된다. 각 turn 의 끝에서 도구 호출 결과가 messages 에 누적되고, 다음 turn 의 시작에서 미들웨어 1\~7 이 다시 위에서 아래로 돈다. 모델은 자기가 미들웨어 안에서 돌고 있는 줄 모른다 — 도구가 그저 거기 있다고만 인식한다.


## §5. 청사진 — `create_deep_agent` 의 다이얼

`create_deep_agent` 는 17개 파라미터를 받지만, 의미상 두 묶음으로 갈라진다.

- **Core Config 세 다이얼**: `model`, `system_prompt`, `tools` — 거의 모든 도메인에서 첫 손이 가는 자리
- **Features 셋**: `backend`, `subagents`, `interrupt_on` — 켜고 끄는 옵션

이 글에서는 좌측 셋에 시간을 쓰고, 우측은 이름만 띄워둔다.


### 그림 8: `create_deep_agent` 의 두 분기

<img src="../content/figs/fig08_blueprint_dials.svg" alt="Blueprint Dials" width="900"/>

### 표 4: Core Config 세 다이얼

| 파라미터 | 기본값 | 무엇이 들어가나 |
|---|---|---|
| `model` | `claude-sonnet-4-6` | LangChain ChatModel 객체 또는 `"provider:model"` 문자열 |
| `system_prompt` | `BASE_AGENT_PROMPT` (Claude Code 영감, ~42줄) | 위에 합성될 도메인 한 장 |
| `tools` | `[]` | 빌트인 9종에 추가될 사용자 도구 리스트 |

### 표 5: Features 3종 — 1주차에서 다루는 깊이

| Feature | 핵심 인자 | 1주차에서 다루는 깊이 |
|---|---|---|
| Backend | `backend=`, `memory=`, `store=` | "Filesystem 의 라우팅 경로를 결정한다" 까지만 (§3.2) |
| Subagents | `subagents=`, `permissions=` | "위임이 가능하다" 까지만 (§3.3) |
| Interrupts | `interrupt_on=` | 이름만 — HITL 패턴은 이 글의 범위 밖 |

> **버전 주의**: 공식 한국어 번역본 [`03-customization_ko.md`](../archives/original_docs/03-customization_ko.md) 는 한 단계 이전 식별자 `claude-sonnet-4-5-20250929` 를 적고 있다. 라이브러리 현재 디폴트는 API 레퍼런스 기준 `claude-sonnet-4-6` 이다.


### §5.1 Demo 2: Model String Swap (스크립트 02) — 길 1

> **한 줄**: `"provider:model"` 문자열 한 줄로 모델을 갈아 끼운다.

`init_chat_model` 의 한 줄 호출 — `OPENAI_BASE_URL` 을 환경변수로 두면 OpenAI 호환 프록시 (clipproxyapi, OpenRouter, Bedrock-OpenAI shim 등) 로 그대로 라우팅된다. 코드 손댈 일 없이 `.env` 만 갈아 끼우면 된다.

**원본**: [`scripts/02_model_string_swap.py`](../archives/scripts_py/02_model_string_swap.py)


In [ ]:
# Demo 2 — Path 1 (문자열로 모델 갈아 끼우기).
#
# Demo 1 에서 만든 model 인스턴스가 이미 "openai:gpt-5.4-mini" 로 만들어져 있다.
# 여기서는 "도구 없이, 시스템 프롬프트도 디폴트 그대로" 인 가장 가벼운 형태를 보인다.
# create_deep_agent() 호출이 끝나면 BASE_AGENT_PROMPT 만 시스템 프롬프트로 들어간다.

agent_demo2 = create_deep_agent(model=model)  # 인자 1개. 정말로 "5줄로 시작" 의 가장 작은 형태.

print(f"agent 타입: {type(agent_demo2).__name__}")
print(f"노드 수: {len(agent_demo2.nodes)}\n")

result_demo2 = agent_demo2.invoke({
    "messages": [{"role": "user", "content": "한 줄로 자기소개 해줘."}]
})

print("=" * 70)
print("응답:")
print("=" * 70)
print(result_demo2["messages"][-1].content)


> **포인트**: 이 코드 자체에 "OpenAI" 도, "clipproxyapi" 도, "gpt-5.4-mini" 도 등장하지 않는다. `.env` 한 줄만 바꾸면 동일 코드가 OpenAI 직접 / 사내 프록시 / 호환 게이트웨이 어디에서든 작동한다. 이 분리가 길 1의 핵심 이점이다.

이제 Ollama (OpenAI 호환이 아닌 백엔드) 로 같은 일을 하려면 어떻게 다르게 짜야 하는지 본다.


### §5.2 Demo 3: Model Object — Ollama (스크립트 03) — 길 2

> **한 줄**: `temperature`, `num_ctx` 같은 세부 다이얼이 필요하거나 OpenAI 호환이 아닌 백엔드(Ollama, Bedrock, Vertex)로 갈 때 이 길로 간다.

**원본**: [`scripts/03_model_object_ollama.py`](../archives/scripts_py/03_model_object_ollama.py)

#### ⚠️ 사전 점검 — Ollama 모델은 tools 를 지원해야 한다

Deep Agent 는 도구 호출(tool calling) 을 끊임없이 한다. **tools 미지원 모델로는 시작조차 못 한다**.

| 모델 | tools 지원 | 비고 |
|---|---|---|
| `llama3:8b`, `llama3.1:8b` | ❌ (`registry.ollama.ai/library/llama3:8b does not support tools`) | 직전 세션에서 400 에러로 확인 |
| `gemma3:12b`, `gemma3:12b-it-qat` | ❌ | gemma3 native tool calling 미지원 |
| `gpt-oss:20b` | ✅ | 20B MoE, OpenAI 가 공개한 오픈 가중치 |
| `PetrosStav/gemma3-tools:12b` | ✅ | gemma3 의 tools 변형 — 본 노트북 기본값 |
| `qwen3-vl:30b-a3b-instruct` | ✅ (튜닝 필요할 수 있음) | 30B vision-language MoE |

`.env` 의 `OLLAMA_MODEL` 값을 위 ✅ 모델 중 하나로 두고 진행한다.

#### ChatOllama 자주 쓰는 파라미터

| 파라미터 | 기본값 | 의미 |
|---|---|---|
| `temperature` | 0.8 | 0 에 가까울수록 결정적, 1 이상은 창의적 |
| `num_ctx` | 2048 | 컨텍스트 윈도우 크기 (Deep Agent 는 이 값을 키우는 게 좋다 — 8192~16384) |
| `num_predict` | 128 | 한 응답에서 생성할 최대 토큰. -1 이면 무제한 |
| `top_p`, `top_k` | 0.9, 40 | 샘플링 분포 제한 |
| `stop` | None | 중단 시퀀스 (예: `["\n\n", "User:"]`) |
| `format` | None | `"json"` 으로 두면 JSON 모드 강제 |


In [ ]:
# Demo 3 — Path 2 (LangChain 모델 객체로 정밀하게).
#
# init_chat_model + provider 명시 → ChatOllama 객체가 반환된다.
# (langchain-ollama 패키지가 .venv 에 설치돼 있어야 한다 — 직전 세션에서 확인 완료)

OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "PetrosStav/gemma3-tools:12b")

model_ollama = init_chat_model(
    model=OLLAMA_MODEL,
    model_provider="ollama",
    temperature=0,        # 결정적 응답
    # 추가 다이얼이 필요하면 여기에 num_ctx=8192 등.
)

print(f"Ollama 모델: {OLLAMA_MODEL}")
print(f"인스턴스: {type(model_ollama).__name__}")

agent_demo3 = create_deep_agent(model=model_ollama)

result_demo3 = agent_demo3.invoke({
    "messages": [{"role": "user", "content": "Ollama, 안녕?"}]
})

print()
print("=" * 70)
print("응답:")
print("=" * 70)
print(result_demo3["messages"][-1].content)


### 그림 (추가): 길 1 vs 길 2 분기

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 360" width="900" style="background:#fff;font-family:'Inter',sans-serif">
  <defs>
    <marker id="arr2" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="#444"/>
    </marker>
  </defs>
  <!-- env -->
  <rect x="40" y="140" width="160" height="80" rx="10" fill="#FEFCE8" stroke="#854D0E" stroke-width="2"/>
  <text x="120" y="170" text-anchor="middle" font-size="13" font-weight="700" fill="#713F12">.env (또는 OS env)</text>
  <text x="120" y="190" text-anchor="middle" font-size="11" fill="#713F12">DEEPAGENT_MODEL</text>
  <text x="120" y="207" text-anchor="middle" font-size="11" fill="#713F12">OPENAI_BASE_URL</text>
  <!-- decision -->
  <polygon points="290,180 360,140 430,180 360,220" fill="#FCE7F3" stroke="#9D174D" stroke-width="2"/>
  <text x="360" y="178" text-anchor="middle" font-size="12" font-weight="700" fill="#831843">길 선택?</text>
  <text x="360" y="194" text-anchor="middle" font-size="10" fill="#831843">속도 vs 정밀도</text>
  <line x1="200" y1="180" x2="290" y2="180" stroke="#444" stroke-width="2" marker-end="url(#arr2)"/>
  <!-- path 1 -->
  <rect x="500" y="50" width="360" height="120" rx="10" fill="#DBEAFE" stroke="#1E40AF" stroke-width="2"/>
  <text x="680" y="80" text-anchor="middle" font-size="14" font-weight="700" fill="#1E3A8A">길 1: provider:model 문자열</text>
  <text x="520" y="105" font-size="11" fill="#1E3A8A" font-family="monospace">init_chat_model(</text>
  <text x="540" y="120" font-size="11" fill="#1E3A8A" font-family="monospace">f"openai:{MODEL_NAME}",</text>
  <text x="540" y="135" font-size="11" fill="#1E3A8A" font-family="monospace">base_url=OPENAI_BASE_URL,</text>
  <text x="520" y="150" font-size="11" fill="#1E3A8A" font-family="monospace">)</text>
  <text x="680" y="165" text-anchor="middle" font-size="11" fill="#1E3A8A" font-style="italic">— Demo 2, Demo 4 (OpenAI 호환)</text>
  <!-- path 2 -->
  <rect x="500" y="190" width="360" height="120" rx="10" fill="#D1FAE5" stroke="#065F46" stroke-width="2"/>
  <text x="680" y="220" text-anchor="middle" font-size="14" font-weight="700" fill="#064E3B">길 2: LangChain 모델 객체</text>
  <text x="520" y="245" font-size="11" fill="#064E3B" font-family="monospace">init_chat_model(</text>
  <text x="540" y="260" font-size="11" fill="#064E3B" font-family="monospace">model=OLLAMA_MODEL,</text>
  <text x="540" y="275" font-size="11" fill="#064E3B" font-family="monospace">model_provider="ollama",</text>
  <text x="540" y="290" font-size="11" fill="#064E3B" font-family="monospace">temperature=0,  # 다이얼</text>
  <text x="520" y="305" font-size="11" fill="#064E3B" font-family="monospace">)</text>
  <line x1="430" y1="170" x2="500" y2="100" stroke="#444" stroke-width="2" marker-end="url(#arr2)"/>
  <line x1="430" y1="190" x2="500" y2="240" stroke="#444" stroke-width="2" marker-end="url(#arr2)"/>
  <text x="450" y="135" font-size="10" font-style="italic" fill="#64748B">속도 우선</text>
  <text x="450" y="225" font-size="10" font-style="italic" fill="#64748B">정밀 우선</text>
  <!-- merge -->
  <rect x="220" y="290" width="500" height="50" rx="10" fill="#FEF3C7" stroke="#92400E" stroke-width="2"/>
  <text x="470" y="320" text-anchor="middle" font-size="13" font-weight="700" fill="#7C2D12">create_deep_agent(model=model)  → CompiledStateGraph</text>
</svg>

> 두 길이 만나는 자리는 동일하다 — `create_deep_agent(model=model)`. 어떤 LangChain ChatModel 인스턴스가 들어와도 deepagents 의 미들웨어 체인은 그 위에서 동일하게 돈다.


### §5.3 Demo 4: Custom System Prompt (스크립트 04)

> **한 줄**: `system_prompt=` 인자는 BASE_AGENT_PROMPT 를 **교체** 하는 게 아니라 그 앞에 **prepend** 된다.

**원본**: [`scripts/04_custom_system_prompt.py`](../archives/scripts_py/04_custom_system_prompt.py)


### 그림 9: 시스템 프롬프트의 3단 합성

<img src="../content/figs/fig09_system_prompt_layers.svg" alt="System Prompt Layers" width="900"/>

매 turn 마다 모델에게 전달되는 최종 시스템 프롬프트는 세 segment 의 합성이다 — `USER → (BASE 또는 CUSTOM) → SUFFIX`, 사이는 빈 줄(`\n\n`)로 결합된다.

| Segment | 출처 | 비어 있을 때 |
|---|---|---|
| **USER** | 사용자가 `system_prompt=` 인자로 넘긴 도메인 한 장 | 없으면 이 segment 는 비워짐 |
| **BASE** | 라이브러리의 `BASE_AGENT_PROMPT` (graph.py 56\~97행). 매칭되는 `HarnessProfile` 이 등록돼 있고 그 프로필이 자체 `base_system_prompt` 를 갖고 있으면 **CUSTOM** 으로 통째 교체 | 거의 항상 차 있음 |
| **SUFFIX** | `HarnessProfile.system_prompt_suffix`. 매칭되는 모델(예: Sonnet 4.6) 이면 끝에 자동 부착되는 XML 태그 묶음(`<use_parallel_tool_calls>` 등) | 매칭 모델 없으면 비워짐 |

**핵심 규칙**: `system_prompt=` 는 BASE 를 **교체하는 게 아니라 그 앞(USER 자리)에 prepend** 된다. 사용자가 도메인 프롬프트를 넘겨도 라이브러리의 BASE 와 모델별 SUFFIX 는 그대로 살아남는다 — Understand → Act → Verify 워크플로와 도구 호출 정책이 사용자 커스텀 한 장에 의해 지워지지 않는 이유다.


In [ ]:
# Demo 4 — 가장 작은 커스텀 system_prompt.
# Quickstart(Demo 1) 의 길고 친절한 prompt 가 도메인을 좁히는 짧은 한 장으로 줄어든 형태일 뿐이다.

research_instructions_short = """\
You are an expert researcher. Your job is to conduct \
thorough research, and then write a polished report. \
"""

agent_demo4 = create_deep_agent(
    model=model,                                # Demo 1 에서 만든 model 재사용
    system_prompt=research_instructions_short,  # 3줄짜리 도메인 프롬프트
)

result_demo4 = agent_demo4.invoke({
    "messages": [
        {"role": "user", "content": "RAG 와 fine-tuning 의 차이를 한 단락으로."}
    ]
})

print("=" * 70)
print("응답 (research_instructions_short 기조로 답변):")
print("=" * 70)
print(result_demo4["messages"][-1].content)


In [ ]:
# 보너스 — BASE_AGENT_PROMPT 실체 들여다보기 (교안 너머).
#
# 교안은 "Claude Code 영감, ~42줄" 이라고만 한다. 실제 어떤 텍스트인지 직접 본다.
# 이 텍스트가 우리가 넘긴 짧은 research_instructions_short 와 빈 줄(\n\n) 로 합성돼
# 모델의 시스템 프롬프트 슬롯에 들어간다.

from deepagents.graph import BASE_AGENT_PROMPT

print("BASE_AGENT_PROMPT 길이:", len(BASE_AGENT_PROMPT), "자")
print("=" * 70)
print(BASE_AGENT_PROMPT)


## §6. 언제 쓰나 — 교안 §5

### 표 6: `create_agent` / 직접 LangGraph / `create_deep_agent` 의사결정

| 상황 | 추천 | 이유 |
|---|---|---|
| 짧은 Q&A, 단일 도구 호출 | `create_agent` | 미들웨어 오버헤드 없이 가볍게 동작 |
| 복잡한 다단계 + 큰 컨텍스트 + 위임/메모리 | `create_deep_agent` | 4대 능력이 즉시 켜짐, BASE 프롬프트로 워크플로 검증됨 |
| 흐름이 결정적이고 노드 통제 필요 | 직접 짠 LangGraph workflow | StateGraph 로 명시 모델링, 분기·병렬·HITL 자유 |
| 짧은 작업이지만 도구 호출 정책이 까다롭다 | `create_agent` + 미들웨어 직접 조립 | 필요한 미들웨어만 골라 끼움 — `create_deep_agent` 의 부분집합 |

> **결정의 한 줄 룰**: **"단계가 5개 이상이고, 컨텍스트가 모델 윈도우의 절반을 넘으며, 부분 작업을 위임할 만하다"** — 이 셋이 동시에 참이면 `create_deep_agent`. 하나만 참이면 다른 길을 검토.


### 그림 (추가): 의사결정 플로우차트

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 480" width="900" style="background:#fff;font-family:'Inter',sans-serif">
  <defs>
    <marker id="arr3" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="#444"/>
    </marker>
  </defs>
  <!-- start -->
  <ellipse cx="450" cy="40" rx="100" ry="22" fill="#E0E7FF" stroke="#3730A3" stroke-width="2"/>
  <text x="450" y="46" text-anchor="middle" font-size="13" font-weight="700" fill="#312E81">에이전트 만들기 시작</text>
  <line x1="450" y1="62" x2="450" y2="90" stroke="#444" stroke-width="2" marker-end="url(#arr3)"/>
  <!-- Q1 -->
  <polygon points="350,100 450,80 550,100 450,140" fill="#FCE7F3" stroke="#9D174D" stroke-width="2"/>
  <text x="450" y="108" text-anchor="middle" font-size="12" font-weight="700" fill="#831843">단계 ≥ 5?</text>
  <text x="450" y="123" text-anchor="middle" font-size="10" fill="#831843">(다단계 작업?)</text>
  <line x1="350" y1="115" x2="200" y2="115" stroke="#444" stroke-width="2" marker-end="url(#arr3)"/>
  <line x1="450" y1="140" x2="450" y2="170" stroke="#444" stroke-width="2" marker-end="url(#arr3)"/>
  <text x="265" y="108" font-size="11" font-style="italic" fill="#475569">No</text>
  <text x="460" y="158" font-size="11" font-style="italic" fill="#475569">Yes</text>
  <!-- Q1-no leaf -->
  <rect x="60" y="100" width="140" height="60" rx="8" fill="#DBEAFE" stroke="#1E40AF" stroke-width="2"/>
  <text x="130" y="125" text-anchor="middle" font-size="12" font-weight="700" fill="#1E3A8A">create_agent</text>
  <text x="130" y="145" text-anchor="middle" font-size="10" fill="#1E3A8A">짧은 Q&amp;A · 단일 호출</text>
  <!-- Q2 -->
  <polygon points="340,180 450,160 560,180 450,220" fill="#FCE7F3" stroke="#9D174D" stroke-width="2"/>
  <text x="450" y="186" text-anchor="middle" font-size="12" font-weight="700" fill="#831843">컨텍스트 &gt; 50%?</text>
  <text x="450" y="201" text-anchor="middle" font-size="10" fill="#831843">(누적 자료가 많은가)</text>
  <line x1="340" y1="195" x2="200" y2="220" stroke="#444" stroke-width="2" marker-end="url(#arr3)"/>
  <line x1="450" y1="220" x2="450" y2="250" stroke="#444" stroke-width="2" marker-end="url(#arr3)"/>
  <text x="265" y="215" font-size="11" font-style="italic" fill="#475569">No</text>
  <text x="460" y="240" font-size="11" font-style="italic" fill="#475569">Yes</text>
  <!-- Q2-no leaf -->
  <rect x="40" y="200" width="180" height="60" rx="8" fill="#FEF3C7" stroke="#92400E" stroke-width="2"/>
  <text x="130" y="225" text-anchor="middle" font-size="12" font-weight="700" fill="#7C2D12">결정적 흐름인가?</text>
  <text x="130" y="245" text-anchor="middle" font-size="10" fill="#7C2D12">→ 직접 짠 LangGraph</text>
  <!-- Q3 -->
  <polygon points="340,260 450,240 560,260 450,300" fill="#FCE7F3" stroke="#9D174D" stroke-width="2"/>
  <text x="450" y="266" text-anchor="middle" font-size="12" font-weight="700" fill="#831843">위임 가능?</text>
  <text x="450" y="281" text-anchor="middle" font-size="10" fill="#831843">(서브 작업 분리?)</text>
  <line x1="340" y1="275" x2="200" y2="320" stroke="#444" stroke-width="2" marker-end="url(#arr3)"/>
  <line x1="450" y1="300" x2="450" y2="340" stroke="#444" stroke-width="2" marker-end="url(#arr3)"/>
  <text x="265" y="305" font-size="11" font-style="italic" fill="#475569">No</text>
  <text x="460" y="330" font-size="11" font-style="italic" fill="#475569">Yes</text>
  <!-- Q3-no leaf -->
  <rect x="40" y="300" width="180" height="60" rx="8" fill="#DBEAFE" stroke="#1E40AF" stroke-width="2"/>
  <text x="130" y="325" text-anchor="middle" font-size="12" font-weight="700" fill="#1E3A8A">create_agent + 미들웨어</text>
  <text x="130" y="345" text-anchor="middle" font-size="10" fill="#1E3A8A">필요한 미들웨어만 골라</text>
  <!-- final leaf -->
  <rect x="340" y="350" width="220" height="80" rx="10" fill="#D1FAE5" stroke="#065F46" stroke-width="3"/>
  <text x="450" y="380" text-anchor="middle" font-size="15" font-weight="700" fill="#064E3B">create_deep_agent</text>
  <text x="450" y="402" text-anchor="middle" font-size="11" fill="#064E3B">4대 능력 즉시 ON</text>
  <text x="450" y="418" text-anchor="middle" font-size="11" fill="#064E3B">+ BASE 프롬프트 검증된 워크플로</text>
  <!-- legend -->
  <rect x="700" y="100" width="180" height="160" rx="8" fill="#F8FAFC" stroke="#94A3B8"/>
  <text x="790" y="120" text-anchor="middle" font-size="11" font-weight="700" fill="#475569">범례</text>
  <rect x="715" y="135" width="20" height="14" fill="#DBEAFE" stroke="#1E40AF"/>
  <text x="745" y="146" font-size="10" fill="#475569">create_agent</text>
  <rect x="715" y="155" width="20" height="14" fill="#FEF3C7" stroke="#92400E"/>
  <text x="745" y="166" font-size="10" fill="#475569">직접 LangGraph</text>
  <rect x="715" y="175" width="20" height="14" fill="#D1FAE5" stroke="#065F46"/>
  <text x="745" y="186" font-size="10" fill="#475569">create_deep_agent</text>
  <rect x="715" y="195" width="20" height="14" fill="#FCE7F3" stroke="#9D174D"/>
  <text x="745" y="206" font-size="10" fill="#475569">의사결정 분기</text>
  <text x="715" y="232" font-size="10" font-style="italic" fill="#64748B">3개 Yes 가 모두</text>
  <text x="715" y="247" font-size="10" font-style="italic" fill="#64748B">참이면 가장 아래로</text>
</svg>

> 이 트리의 "Yes 3연타" 가 1주차 한 줄 룰의 시각화다. 단계가 길고, 자료가 컨텍스트를 잡아먹고, 무거운 부분을 위임할 만하면 `create_deep_agent` 가 가장 빠르게 일을 맺는다.


## §7. 교안 너머 — 보조 정보

### §7.1 이 글에서 다루지 않은 것 (다음 발표자들)

| 주제 | 다음 발표자 | 다뤄지는 것 |
|---|---|---|
| **컨텍스트 · 메모리 · 스킬** | 정훈 | `CompositeBackend` 의 prefix 라우팅 디테일, LangGraph Store 영속화 모델, `skills=` 파라미터 |
| **백엔드 · 샌드박스 · 권한** | 종훈L | Subagents 의 권한 격리, Shell tool / Context editing, `permissions=` 파라미터, sandbox 가 코드 실행을 격리하는 방식 |
| **Middleware Architecture** | (별도 글) | 16종 prebuilt middleware 의 분류, 직접 미들웨어 작성, before/after model hook |
| **HITL (Human-in-the-Loop)** | (별도 글) | `interrupt_on=` 으로 도구 호출 사이 사람 승인 받기, Command resume 패턴 |
| **CLI · LangSmith Deployment** | (별도 글) | `deepagents-cli` 로 에이전트를 CLI 로 배포, LangSmith 에 그래프 publish |

### §7.2 LangSmith 트레이싱 — 한 번에 켜기

`LANGCHAIN_TRACING_V2=true` 와 `LANGCHAIN_API_KEY=...` 두 환경변수만 있으면 `create_deep_agent()` 가 만든 그래프의 모든 turn 이 LangSmith UI 에 자동 기록된다 — 메시지·도구 호출·토큰 사용량·시간까지. 별도 코드 변경은 필요 없다.

> 본 노트북에서는 LangSmith API 키를 가정하지 않으니 시범 코드만 보여주고 실제 호출은 하지 않는다. 사용 시 [smith.langchain.com](https://smith.langchain.com/) 에서 키를 발급해 `.env` 에 추가.


In [ ]:
# (실행 안 함) LangSmith 트레이싱 활성화 시범.
#
# 아래 두 줄을 .env 에 추가하기만 하면 다음 invoke 부터 자동 추적된다.
# (실제 키가 없으면 LangSmith 가 조용히 무시하므로 setenv 만 보여준다.)

import os

os.environ.setdefault("LANGCHAIN_TRACING_V2", "false")  # true 로 두면 추적 시작
os.environ.setdefault("LANGCHAIN_API_KEY", "")          # ls__... 로 시작하는 LangSmith 키
os.environ.setdefault("LANGCHAIN_PROJECT", "week1-walkthrough")  # 프로젝트 라벨

print(f"LANGCHAIN_TRACING_V2 = {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"LANGCHAIN_PROJECT    = {os.environ['LANGCHAIN_PROJECT']}")
print(f"API_KEY 설정?         = {bool(os.environ['LANGCHAIN_API_KEY'])}")
print()
print("→ 위 세 값이 모두 설정된 상태에서 agent.invoke(...) 를 호출하면,")
print("   LangSmith UI 의 'week1-walkthrough' 프로젝트에 trace 가 쌓인다.")


### §7.3 LangChain 통합 모델 — 패키지 / 문자열 스펙 표

| 제공자 | 문자열 스펙 (`provider:model`) | 패키지 | 비고 |
|---|---|---|---|
| OpenAI (직접) | `openai:gpt-5`, `openai:gpt-4o-mini` | `langchain-openai` | API 키 필요 |
| OpenAI 호환 프록시 | `openai:<proxy-model>` + `base_url=` | `langchain-openai` | clipproxyapi, OpenRouter, vLLM-OpenAI shim 등 |
| Anthropic | `anthropic:claude-sonnet-4-6` | `langchain-anthropic` | deepagents 의 디폴트 모델 |
| Google Gemini | `google_genai:gemini-2.0-flash` | `langchain-google-genai` | tool calling 지원 |
| AWS Bedrock | `bedrock_converse:<model_id>` | `langchain-aws` | Anthropic/Mistral/Titan 등 |
| GCP Vertex AI | `vertexai:<model>` | `langchain-google-vertexai` | Gemini, Mistral 등 |
| Azure OpenAI | `azure_openai:<deployment>` | `langchain-openai` (azure 모드) | endpoint + api_version 필요 |
| Ollama (로컬) | `ollama:<model>` | `langchain-ollama` | tools 지원 모델만 가능 |
| Together AI | `together:<model>` | `langchain-together` | 오픈 가중치 호스팅 |
| Groq | `groq:<model>` | `langchain-groq` | Llama 3.x · Mixtral · Gemma2 등 |

> 모든 제공자는 `init_chat_model("provider:model", **kwargs)` 한 줄로 시작하면 된다. 표에 없는 통합은 [LangChain Chat Models 카탈로그](https://docs.langchain.com/oss/python/integrations/chat) 에서 확인.


## §8. 용어집 — 교안 부록 A 발췌

| 용어 | 정의 |
|---|---|
| **Deep Agent** / **Deep Agents** | LangGraph 위에 4대 능력을 미들웨어로 내장한 에이전트 (개념) / 라이브러리 |
| `deepagents` | PyPI 패키지명 (소문자 그대로) |
| `create_deep_agent()` | 라이브러리의 메인 팩토리 함수 — `CompiledStateGraph` 반환 |
| `BASE_AGENT_PROMPT` | 기본 시스템 프롬프트 (Claude Code 영감, ~42줄). `deepagents.graph` 모듈에 정의 |
| 미들웨어 (middleware) | LangChain core 가 제공하는 prebuilt 16종 + Deep Agents 전용 (Filesystem, Subagent, Skills 등) |
| Backend | Filesystem 미들웨어의 저장 추상 — `StateBackend` (기본) / `StoreBackend` / `CompositeBackend` |
| Store | LangGraph 의 thread 횡단 영속 계층 — 장기 메모리의 기반 |
| Subagent | 격리된 컨텍스트·권한으로 일하는 하위 에이전트 — `task` 도구로 호출 |
| `general-purpose` subagent | 디폴트로 항상 존재하는 범용 서브에이전트 — 메인과 동일한 도구 풀, 컨텍스트 격리만 제공 |
| HarnessProfile | 모델별 프롬프트·미들웨어 튜닝 묶음 — 매칭 모델일 때 BASE 자리를 자체 `base_system_prompt` 로 바꾸고 끝에 `system_prompt_suffix` 부착 |
| Tool calling | 모델이 도구 함수를 자율적으로 호출하는 능력. Deep Agent 의 모든 동작이 이 위에 서 있다 (지원 안 하는 모델로는 구동 불가) |
| `CompiledStateGraph` | LangGraph 의 컴파일된 상태 그래프 — `create_deep_agent()` 의 반환 타입. `.invoke / .stream / .ainvoke` 모두 작동 |
| Checkpointer | LangGraph 의 한 thread 내 영속 계층 — `agent.invoke` 의 중간 상태를 저장해 다음 호출에서 이어가게 한다 |
| CodeAct | 도구 호출 대신 Python 스크립트를 생성·실행하는 패러다임 (Manus) |
| ReAct | Reason → Act → Observe 의 한 사이클로 도는 가장 단순한 에이전트 패턴. `create_agent` 의 기반 |


## §9. 레퍼런스 & 추가 자료

### 9.1 공식 문서

| 문서 | 링크 |
|---|---|
| Deep Agents Overview | <https://docs.langchain.com/oss/python/deepagents/overview> |
| Deep Agents Quickstart | <https://docs.langchain.com/oss/python/deepagents/quickstart> |
| Customization | <https://docs.langchain.com/oss/python/deepagents/customization> |
| Middleware | <https://docs.langchain.com/oss/python/deepagents/middleware> |
| CLI | <https://docs.langchain.com/oss/python/deepagents/cli> |
| API Reference | <https://reference.langchain.com/python/deepagents/> |

### 9.2 LangChain · LangGraph

| 문서 | 링크 |
|---|---|
| LangChain Agents | <https://docs.langchain.com/oss/python/langchain/agents> |
| LangGraph Overview | <https://docs.langchain.com/oss/python/langgraph/overview> |
| LangGraph Deploy | <https://docs.langchain.com/oss/python/langgraph/deploy> |
| Tool Calling Concepts | <https://docs.langchain.com/oss/python/concepts/tool-calling> |
| Chat Model Integrations | <https://docs.langchain.com/oss/python/integrations/chat> |

### 9.3 LangSmith (관측 · 평가 · 배포)

| 문서 | 링크 |
|---|---|
| LangSmith | <https://smith.langchain.com/> |
| LangSmith Observability | <https://docs.smith.langchain.com/observability> |
| LangSmith Deployment | <https://docs.smith.langchain.com/deployment> |

### 9.4 PyPI · GitHub · 출시 발표

| 자료 | 링크 |
|---|---|
| `deepagents` PyPI | <https://pypi.org/project/deepagents/> |
| `langchain-ai/deepagents` GitHub | <https://github.com/langchain-ai/deepagents> |
| Harrison Chase 출시 발표 (LangChain Blog) | <https://blog.langchain.com/deep-agents/> |

### 9.5 영감 시스템

| 시스템 | 링크 |
|---|---|
| Claude Code (Anthropic Engineering) | <https://www.anthropic.com/engineering/claude-code> |
| OpenAI Deep Research | <https://openai.com/index/introducing-deep-research/> |
| Manus | <https://manus.im/> |
| Anthropic Models Overview | <https://platform.claude.com/docs/en/about-claude/models/overview> |

### 9.6 검색 · 모델 (실행 환경)

| 자료 | 링크 |
|---|---|
| Tavily (검색 API) | <https://tavily.com/> |
| Ollama | <https://ollama.com/> |
| Ollama tool-calling 모델 카탈로그 | <https://ollama.com/search?c=tools> |

### 9.7 본 프로젝트 내부 문서

| 문서 | 경로 |
|---|---|
| 1주차 교안 | [`content/01_textbook.md`](../archives/source/01_textbook.md) |
| 1주차 슬라이드 (HTML) | [`content/slide.html`](../archives/source/slide.html) |
| 1주차 슬라이드 (PDF) | [`content/slide.pdf`](../content/slides.pdf) |
| 1주차 리포트 (PDF) | [`content/01_textbook.pdf`](../content/textbook.pdf) |
| 한국어 원본 — 개요 | [`01-overview_ko.md`](../archives/original_docs/01-overview_ko.md) |
| 한국어 원본 — 빠른 시작 | [`02-quickstart_ko.md`](../archives/original_docs/02-quickstart_ko.md) |
| 한국어 원본 — 커스터마이징 | [`03-customization_ko.md`](../archives/original_docs/03-customization_ko.md) |
| 9주 학습 계획 | [`20260502_DeepAgent_Study_Plan.md`](../../20260502_DeepAgent_Study_Plan.md) |


## §10. 부록 — 실행 환경 메모

### 10.1 venv 활성화

```bash
# 본 노트북이 가정하는 .venv 위치
source /home/restful3/workspace/langchain-docs/deep-agents/.venv/bin/activate

# 또는 Jupyter 안에서는 커널 선택만 하면 됨
# (커널 이름: Python (.venv) 또는 사용자가 등록한 displayname)
```

### 10.2 `.env` 키 의미

| 키 | 의미 | 본 노트북 기본값 |
|---|---|---|
| `OPENAI_API_KEY` | OpenAI 또는 OpenAI 호환 프록시의 키 | clipproxyapi 키 (`your-api-key-1`) |
| `OPENAI_BASE_URL` | OpenAI 호환 프록시 endpoint. 비우면 OpenAI 직접 | `http://localhost:8317/v1` (clipproxyapi 로컬) |
| `DEEPAGENT_MODEL` | 사용할 모델 (`provider:` 부분 제외) | `gpt-5.4-mini` (clipproxyapi 노출 모델) |
| `TAVILY_API_KEY` | Tavily 검색 API 키 (Demo 1 전용) | `tvly-dev-...` |
| `OLLAMA_MODEL` | Ollama 모델 (Demo 3 전용). tools 지원 모델이어야 함 | `PetrosStav/gemma3-tools:12b` |

### 10.3 트러블슈팅

| 증상 | 원인 / 해결 |
|---|---|
| `KeyError: 'TAVILY_API_KEY'` | `.env` 에 `TAVILY_API_KEY` 가 비어 있음 (Demo 1 전용) — `https://tavily.com/` 에서 발급 |
| `openai.AuthenticationError 401` | `OPENAI_API_KEY` 잘못됐거나 `OPENAI_BASE_URL` 과 짝이 안 맞음 — clipproxyapi 키가 OpenAI 공식 endpoint 로 흘러갔거나 그 반대 |
| `httpx.ConnectError` (clipproxyapi) | 프록시 데몬 (포트 8317) 안 떠 있음 — `lsof -i :8317` 로 확인 |
| `httpx.ConnectError` (Ollama) | `ollama serve` 안 돌고 있음 — `curl http://localhost:11434/api/tags` 로 확인 |
| `ImportError: langchain_ollama` | `.venv/bin/pip install langchain-ollama` |
| `ResponseError: <model> does not support tools` | Ollama 모델이 tool calling 미지원 — `OLLAMA_MODEL` 을 `PetrosStav/gemma3-tools:12b`, `gpt-oss:20b`, `qwen3-vl:30b-a3b-instruct` 중 하나로 |
| `RecursionError` 또는 도구 호출 무한 루프 | 모델이 tool calling 을 잘 못 하는 약한 모델임 — 더 강한 모델(`gpt-5.4-mini`, `gpt-oss:20b`)로 |
| 응답이 한국어 끝까지 안 나옴 | num_predict 가 작거나 모델이 stop token 을 잘못 잡음 — `init_chat_model(..., num_predict=-1)` |

### 10.4 본 노트북 재실행 시 체크리스트

1. `.venv` 활성화 또는 커널을 `.venv` 로 선택
2. `.env` 의 5개 키가 채워져 있는가 (§10.2)
3. clipproxyapi 가 떠 있는가 (Demo 1 / 2 / 4)
4. Ollama 가 떠 있고 `OLLAMA_MODEL` 모델이 `pull` 돼 있는가 (Demo 3)
5. §1 환경 점검 셀이 모두 OK 로 통과하는가

이 5단계가 통과하면 4개 데모가 모두 end-to-end 로 돈다 — 직전 세션에서 검증 완료.

---

> **다음 주 예고**: 정훈 — 컨텍스트 · 메모리 · 스킬 (`CompositeBackend`, LangGraph Store, `skills=` 파라미터)
